# Package

In [1]:
# ----------------------------
# Core
# ----------------------------
from pathlib import Path
import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
import pickle

# ----------------------------
# Feature Store
# ----------------------------
from feast import FeatureStore

# Model
from statsmodels.tsa.ar_model import AutoReg
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# MFLOW
import mlflow

# Importation des données

In [2]:
# ----------------------------
# Locate Feast repo (notebook-safe)
# ----------------------------
def find_project_root(start: Path, marker: str = "2_data_processing") -> Path:
    p = start.resolve()
    for parent in [p] + list(p.parents):
        if (parent / marker).exists():
            return parent
    raise FileNotFoundError(
        f"Impossible de trouver la racine projet (marker '{marker}') depuis {start}"
    )

PROJECT_ROOT = find_project_root(Path.cwd(), marker="2_data_processing")

FEAST_REPO_PATH = (
    PROJECT_ROOT
    / "2_data_processing"
    / "feature_store"
    / "feast_repo"
    / "feature_repo"
)

print("FEAST_REPO_PATH:", FEAST_REPO_PATH)
print("feature_store.yaml exists:", (FEAST_REPO_PATH / "feature_store.yaml").exists())


# ----------------------------
# Load features from Feast
# ----------------------------
def load_features_from_feast(entity_df: pd.DataFrame, feature_refs: list[str]) -> pd.DataFrame:
    fs = FeatureStore(repo_path=str(FEAST_REPO_PATH))
    return fs.get_historical_features(entity_df=entity_df, features=feature_refs).to_df()


# ----------------------------
# Config data
# ----------------------------
START = "1959-01-01"
END   = "2025-09-01"
FREQ  = "MS"
SERIES_ID = "UNRATE"
FEATURE_REFS = ["stationary_value:value"]  # UNRATE déjà stationnaire

dates = pd.date_range(start=START, end=END, freq=FREQ)
entity_df = pd.DataFrame({"series_id": [SERIES_ID] * len(dates), "date": dates})

ts_raw = load_features_from_feast(entity_df, FEATURE_REFS)

ts = (
    ts_raw
    .rename(columns={"series_id": "unique_id", "date": "ds", "value": "y"})
    .sort_values(["unique_id", "ds"])
    .reset_index(drop=True)
)

FEAST_REPO_PATH: D:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\2_data_processing\feature_store\feast_repo\feature_repo
feature_store.yaml exists: True
Using date as the event timestamp. To specify a column explicitly, please name it event_timestamp.


# Préparation des données

## Passage en format WIDE

In [3]:
ts_raw = (
    ts_raw
    .pivot(index="date", columns="series_id", values="value")
    .sort_index()
)

print(ts_raw)

series_id                  UNRATE
date                             
1960-01-01 00:00:00+00:00    -0.8
1960-02-01 00:00:00+00:00    -1.1
1960-03-01 00:00:00+00:00    -0.2
1960-04-01 00:00:00+00:00     0.0
1960-05-01 00:00:00+00:00     0.0
...                           ...
2025-05-01 00:00:00+00:00     0.2
2025-06-01 00:00:00+00:00     0.0
2025-07-01 00:00:00+00:00     0.0
2025-08-01 00:00:00+00:00     0.1
2025-09-01 00:00:00+00:00     0.3

[789 rows x 1 columns]


## Construire la série y

In [4]:
# Vérifie que l’index est bien une date (sinon essaie de le convertir)
y = ts_raw.copy()

if not isinstance(y.index, (pd.DatetimeIndex, pd.PeriodIndex)):
    y.index = pd.to_datetime(y.index, errors="coerce")

# aménager la fréquence mensuelle (début de mois)
y.index = y.index.to_period("M").to_timestamp(how="start")
y = y.sort_index().asfreq("MS").astype(float)

# 🔒 borne la date max (sans dropna)
y = y.loc[:pd.Timestamp("2025-08-01")]

# y (DataFrame) → Series 1D
if isinstance(y, pd.DataFrame):
    y = y.iloc[:, 0]

print(
    f"✅ Série prête : {y.index.min().date()} → {y.index.max().date()} "
    f"| n={len(y)} | freq={y.index.freqstr}"
)

✅ Série prête : 1960-01-01 → 2025-08-01 | n=788 | freq=MS


C:\Users\Mita\AppData\Local\Temp\ipykernel_1288\4267632309.py:8: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  y.index = y.index.to_period("M").to_timestamp(how="start")


In [5]:
y

date
1960-01-01   -0.8
1960-02-01   -1.1
1960-03-01   -0.2
1960-04-01    0.0
1960-05-01    0.0
             ... 
2025-04-01    0.3
2025-05-01    0.2
2025-06-01    0.0
2025-07-01    0.0
2025-08-01    0.1
Freq: MS, Name: UNRATE, Length: 788, dtype: float64

# Run_config

In [6]:

# ---------- Paramètres ----------
h = 12
min_train_n = 36
trend = "c"
p_fixed = 1

# ---------- Paramètres bagging ----------
use_bagging = True
B_boot = 30
L_block = 12
rng = np.random.default_rng(123)

# ---------- Paramètres conformal (comme code 1) ----------
use_conformal = True
step_size = 12
pi_windows = 3
alpha = 0.05  # 95% => q = quantile 1-alpha des erreurs

## Boostrap Utilities

In [7]:
# =========================
# cellule 2 : Bootstrap + fonctions Conformal
# =========================
def moving_block_bootstrap(arr, L, rng):
    """Concatène des blocs contigus de taille L tirés aléatoirement jusqu'à longueur n."""
    n = len(arr)
    if L <= 0 or L > n:
        raise ValueError("L_block invalide")
    nb = int(np.ceil(n / L))
    starts = rng.integers(0, n - L + 1, size=nb)
    out = np.concatenate([arr[s:s+L] for s in starts])[:n]
    return out

def bagged_h_forecast_AR1(y_tr, h, trend, B, L, rng):
    """
    Prévision à horizon h par bagging (residual moving-block bootstrap) pour AR(1).
    Retourne (yhat_mean, yhat_dist, base_pred)
    """
    base_model = AutoReg(y_tr, lags=1, old_names=False, trend=trend).fit()
    base_fc = base_model.predict(start=len(y_tr), end=len(y_tr) + h - 1)
    base_pred = float(base_fc.iloc[-1])

    resid = base_model.resid.values
    fitted = (y_tr.iloc[-len(resid):].values - resid)  # ŷ_t aligné aux résidus

    boot_preds = []
    for _ in range(B):
        res_b = moving_block_bootstrap(resid, L, rng)
        y_b = fitted + res_b
        m_b = AutoReg(pd.Series(y_b, index=y_tr.index[-len(y_b):]),
                      lags=1, old_names=False, trend=trend).fit()
        fc_b = m_b.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        boot_preds.append(float(fc_b.iloc[-1]))

    return float(np.mean(boot_preds)), np.array(boot_preds), base_pred

def fit_predict_ar_p(y_tr, h, trend="c", p=1):
    """Fit AR(p) sur y_tr, retourne la prévision au pas h (dernier point)."""
    m = AutoReg(y_tr, lags=p, old_names=False, trend=trend).fit()
    fc = m.predict(start=len(y_tr), end=len(y_tr) + h - 1)
    return float(fc.iloc[-1])

def conformal_q_from_past_windows_pos(y, i_end, *, h=12, step_size=12, pi_windows=3, trend="c", p=1, alpha=0.05):
    """
    Version robuste (sans test 'date in index'):
    - i_end = position de t_end dans y.index
    - fenêtres de calibration: i_cal = i_end - k*step_size
    - cible à comparer: i_cal + h
    - erreurs: |y[i_cal+h] - yhat( train jusqu'à i_cal )|
    """
    errs = []
    for k in range(1, pi_windows + 1):
        i_cal = i_end - k * step_size
        i_cal_fore = i_cal + h
        if i_cal < 0 or i_cal_fore >= len(y):
            continue

        y_tr_cal = y.iloc[: i_cal + 1]
        if len(y_tr_cal) < max(36, p + 2):  # cohérent avec min_train_n
            continue

        yhat_cal = fit_predict_ar_p(y_tr_cal, h=h, trend=trend, p=p)
        err = abs(float(y.iloc[i_cal_fore]) - yhat_cal)
        errs.append(err)

    if len(errs) == 0:
        return np.nan

    # intervalle symétrique: q = quantile(1-alpha) des erreurs absolues
    return float(np.quantile(errs, 1 - alpha))

In [8]:
# =========================
# cellule 3 : Sécurisation de la série y (index strictement début de mois)
# =========================
y = pd.Series(y.astype(float).values, index=pd.to_datetime(y.index))

# Aligne sur le 1er du mois (start of month) de façon robuste
y.index = y.index.to_period("M").to_timestamp(how="start")

# Fixe la fréquence MS (Month Start)
y = y.asfreq("MS").dropna()

print(f"y: {y.index.min().date()} → {y.index.max().date()}  (n={len(y)}) | freq={y.index.freqstr}")

y: 1960-01-01 → 2025-08-01  (n=788) | freq=MS


In [9]:
# =========================
# cellule 4 : Boucle pseudo-OOS continue + prédiction + intervalles (conformal ou bootstrap)
# =========================
rows = []
last_model = None
last_fit_end = None

# on boucle uniquement sur des t_end qui ont un t_end+h existant
for i_end, t_end in enumerate(y.index[:-h]):
    y_tr = y.iloc[: i_end + 1]
    if len(y_tr) < max(min_train_n, p_fixed + 1):
        continue

    # fit AR(p) base
    ar1 = AutoReg(y_tr, lags=p_fixed, old_names=False, trend=trend).fit()
    last_model = ar1
    last_fit_end = t_end

    # ----- Prévision à h mois (bagging ou base) -----
    if use_bagging:
        yhat_h, yhat_dist, yhat_h_base = bagged_h_forecast_AR1(
            y_tr=y_tr, h=h, trend=trend, B=B_boot, L=L_block, rng=rng
        )
    else:
        fc = ar1.predict(start=len(y_tr), end=len(y_tr) + h - 1)
        yhat_h = float(fc.iloc[-1])
        yhat_h_base = yhat_h
        yhat_dist = None

    # t_fore pris DIRECTEMENT depuis l'index (robuste)
    t_fore = y.index[i_end + h]
    y_true = float(y.iloc[i_end + h])

    # ----- Intervalles -----
    if use_conformal:
        q = conformal_q_from_past_windows_pos(
            y=y, i_end=i_end, h=h, step_size=step_size, pi_windows=pi_windows,
            trend=trend, p=p_fixed, alpha=alpha
        )
        if np.isfinite(q):
            yhat_p05 = yhat_h - q
            yhat_p95 = yhat_h + q
        else:
            yhat_p05 = np.nan
            yhat_p95 = np.nan
    else:
        # fallback: quantiles bootstrap (si bagging), sinon NA
        if use_bagging and (yhat_dist is not None) and len(yhat_dist) > 0:
            yhat_p05 = float(np.percentile(yhat_dist, 5))
            yhat_p95 = float(np.percentile(yhat_dist, 95))
        else:
            yhat_p05 = np.nan
            yhat_p95 = np.nan

    rows.append((t_fore, yhat_h, y_true, yhat_p05, yhat_p95, yhat_h_base))

In [10]:
# =========================
# cellule 5 : DataFrame OOS
# =========================
if rows:
    df_oos_ar1 = (
        pd.DataFrame(
            rows,
            columns=["date", "y_hat", "y_true", "y_hat_p05", "y_hat_p95", "y_hat_base"]
        )
        .set_index("date")
        .sort_index()
    )
else:
    df_oos_ar1 = pd.DataFrame(columns=["y_hat", "y_true", "y_hat_p05", "y_hat_p95", "y_hat_base"])
    df_oos_ar1.index = pd.to_datetime(pd.Index([]))

print(f"\n✅ Pseudo-OOS terminé — n prévisions = {len(df_oos_ar1)}")
print(df_oos_ar1.head(-3))


✅ Pseudo-OOS terminé — n prévisions = 741
               y_hat  y_true  y_hat_p05  y_hat_p95  y_hat_base
date                                                          
1963-12-01  0.070473     0.0        NaN        NaN   -0.080890
1964-01-01  0.017682    -0.1        NaN        NaN    0.141077
1964-02-01  0.090722    -0.5        NaN        NaN    0.408114
1964-03-01  0.165681    -0.3        NaN        NaN    0.242637
1964-04-01  0.091963    -0.4        NaN        NaN    0.238955
...              ...     ...        ...        ...         ...
2025-01-01 -0.010382     0.3  -3.300127   3.279363    0.043861
2025-02-01 -0.006875     0.2  -3.275618   3.261868    0.072590
2025-03-01 -0.019816     0.3  -2.895811   2.856179    0.101443
2025-04-01  0.005002     0.3  -0.589461   0.599465    0.130432
2025-05-01 -0.003852     0.2  -0.617284   0.609580    0.102350

[738 rows x 5 columns]


In [11]:
# ---------- (facultatif) Scores par période ----------
if len(df_oos_ar1):
    df_val  = df_oos_ar1.loc["1983-01-01":"1989-12-31"].copy()
    df_test = df_oos_ar1.loc["1990-01-01":"2025-08-31"].copy()

    if len(df_val):
        mae  = mean_absolute_error(df_val["y_true"], df_val["y_hat"])
        rmse = np.sqrt(mean_squared_error(df_val["y_true"], df_val["y_hat"]))
        r2   = r2_score(df_val["y_true"], df_val["y_hat"]) if len(df_val) > 1 else np.nan
        print(f"\n📊 Validation 83–89 — n={len(df_val)} | MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.3f}")

    if len(df_test):
        mae  = mean_absolute_error(df_test["y_true"], df_test["y_hat"])
        rmse = np.sqrt(mean_squared_error(df_test["y_true"], df_test["y_hat"]))
        r2   = r2_score(df_test["y_true"], df_test["y_hat"]) if len(df_test) > 1 else np.nan
        print(f"📊 Test 90–2025 — n={len(df_test)} | MAE={mae:.3f} | RMSE={rmse:.3f} | R²={r2:.3f}")


📊 Validation 83–89 — n=84 | MAE=0.817 | RMSE=1.234 | R²=-0.949
📊 Test 90–2025 — n=428 | MAE=0.867 | RMSE=1.600 | R²=-0.100


In [12]:
# ==========================================
# Sauvegardes — AR(1) bagging + conformal (h=12)
# ==========================================
import joblib
import pickle

AR1_LAST_PKL  = "AR1_last_trained_model.pkl"
AR1_LAST_META = "AR1_last_trained_model_meta.csv"
AR1_BUNDLE    = "AR1_h12_oos_bundle.pkl"

In [13]:
if last_model is not None:
    try:
        joblib.dump(last_model, AR1_LAST_PKL)
        print(f"💾 Modèle AR(1) sauvegardé → {AR1_LAST_PKL}")
    except Exception:
        with open(AR1_LAST_PKL, "wb") as f:
            pickle.dump(last_model, f)
        print(f"💾 Modèle AR(1) sauvegardé (pickle) → {AR1_LAST_PKL}")

💾 Modèle AR(1) sauvegardé → AR1_last_trained_model.pkl


In [14]:
bundle = {
    # ---------- Prévisions OOS ----------
    "oos_predictions": (
        df_oos_ar1
        .reset_index()
        .rename(columns={
            "y_hat": "y_pred",
            "y_true": "y_obs"
        })
        .assign(
            date=lambda d: pd.to_datetime(d["date"])
                            .dt.to_period("M")
                            .dt.to_timestamp(how="start")
        )
    ),

    # ---------- Paramètres du modèle ----------
    "params": {
        "model": "AR(1)",
        "trend": trend,
        "horizon": h,
        "lags": p_fixed,
        "min_train_n": min_train_n,

        # ---- bagging ----
        "use_bagging": bool(use_bagging),
        "B_boot": int(B_boot),
        "L_block": int(L_block),

        # ---- conformal ----
        "use_conformal": bool(use_conformal),
        "pi_windows": int(pi_windows),
        "step_size": int(step_size),
        "alpha": float(alpha),
    },

    # ---------- Métadonnées ----------
    "meta": {
        "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
        "index_freq": "MS",
        "n_obs_y": int(len(y)),
        "n_forecasts": int(len(df_oos_ar1)),
        "n_intervals_available": int(df_oos_ar1["y_hat_p05"].notna().sum()),
        "first_interval_date": (
            str(df_oos_ar1["y_hat_p05"].first_valid_index().date())
            if df_oos_ar1["y_hat_p05"].notna().any()
            else None
        ),
    }
}

with open(AR1_BUNDLE, "wb") as f:
    pickle.dump(bundle, f)

print(f"💾 Bundle AR(1) OOS sauvegardé → {AR1_BUNDLE}")


💾 Bundle AR(1) OOS sauvegardé → AR1_h12_oos_bundle.pkl


In [15]:
meta_row = {
    "model": "AR(1)",
    "trend": trend,
    "horizon": h,
    "lags": p_fixed,
    "trained_until": str(last_fit_end.date()) if last_fit_end is not None else None,
    "n_obs_y": int(len(y)),
    "n_forecasts": int(len(df_oos_ar1)),
    "use_bagging": bool(use_bagging),
    "use_conformal": bool(use_conformal),
    "pi_windows": pi_windows if use_conformal else None,
    "step_size": step_size if use_conformal else None,
    "n_intervals_available": int(df_oos_ar1["y_hat_p05"].notna().sum()),
}

pd.DataFrame([meta_row]).to_csv(AR1_LAST_META, index=False)
print(f"💾 Méta AR(1) sauvegardée → {AR1_LAST_META}")

💾 Méta AR(1) sauvegardée → AR1_last_trained_model_meta.csv


# Graphique

In [16]:
# =========================
# cellule 7 : Préparer df_obs + df_fcst (format utilsforecast)
# =========================
from utilsforecast.plotting import plot_series

SERIES_ID = "UNRATE"  # adapte si tu veux (ex: "UNRATE_stationary")

# df_obs : observations
df_obs = (
    df_oos_ar1
    .reset_index()
    .rename(columns={"date": "ds", "y_true": "y"})
    .assign(unique_id=SERIES_ID)
    [["unique_id", "ds", "y"]]
)

# df_fcst : forecasts + intervalles
df_fcst = (
    df_oos_ar1
    .reset_index()
    .rename(columns={"date": "ds", "y_hat": "AR1", "y_hat_p05": "AR1-lo-95", "y_hat_p95": "AR1-hi-95"})
    .assign(unique_id=SERIES_ID)
    [["unique_id", "ds", "AR1", "AR1-lo-95", "AR1-hi-95"]]
)

print(df_obs.head(2))
print(df_fcst.head(2))

  unique_id         ds    y
0    UNRATE 1963-12-01  0.0
1    UNRATE 1964-01-01 -0.1
  unique_id         ds       AR1  AR1-lo-95  AR1-hi-95
0    UNRATE 1963-12-01  0.070473        NaN        NaN
1    UNRATE 1964-01-01  0.017682        NaN        NaN


In [40]:
import pandas as pd
import plotly.graph_objects as go
from utilsforecast.plotting import plot_series

# =========================
# cellule 9 : Zoom + cadrage de l’axe Y
# =========================
START_ZOOM = "1990-01-01"
END_ZOOM   = "2025-08-01"

segments = [
    ("1990-01-01", "1999-12-31", "1990-1999"),
    ("2000-01-01", "2008-12-31", "2000-2008"),
    ("2008-01-01", "2019-12-31", "2008-2019"),
    ("2020-01-01", None,         "2019-fin"),
]

# Sécurité datetime
df_obs["ds"] = pd.to_datetime(df_obs["ds"])
df_fcst["ds"] = pd.to_datetime(df_fcst["ds"])

start_zoom_dt = pd.to_datetime(START_ZOOM)
end_zoom_dt   = pd.to_datetime(END_ZOOM)

df_obs_z = df_obs.query("ds >= @start_zoom_dt and ds <= @end_zoom_dt")
df_fcst_z = df_fcst.query("ds >= @start_zoom_dt and ds <= @end_zoom_dt")

# =========================
# Figure de base
# =========================
fig = plot_series(
    df=df_obs_z,
    forecasts_df=df_fcst_z,
    level=[95],
    engine="plotly",
).update_layout(
    height=450,
    yaxis=dict(range=[-10, 12]),
    xaxis=dict(  # Ajout pour agrandir les étiquettes de l'axe x
        tickfont=dict(  # Modification de la police des ticks
            size=14,  # Taille agrandie (changez à 16 ou plus si besoin)
            color="black"  # Couleur par défaut, ajustez si nécessaire
        )
    ),
    title="Unemployment Forecasting",
)

# Supprimer l’annotation "unique_id=UNRATE"
fig.layout.annotations = tuple(
    a for a in fig.layout.annotations
    if "unique_id=" not in a.text
)

# =========================
# Renommage de la légende
# =========================
for trace in fig.data:
    name = trace.name.lower()
    if trace.name == "y":
        trace.name = "Unemployment rate growth"
    elif trace.name == "AR1":
        trace.name = "AR(1) forecast"
    elif "95" in name or "lo" in name or "hi" in name:
        trace.name = "Conformal Prediction"

# =========================
# Un seul bouton "Segments" (sans 1990)
# =========================
ymin, ymax = -10, 12
SEG_GROUP = "SEGMENTS"

fig.update_layout(legend=dict(groupclick="togglegroup"))

first = True
for i, (start, _, _) in enumerate(segments):

    # on ignore le premier segment (1990)
    if i == 0:
        continue

    x = pd.to_datetime(start)

    if not (start_zoom_dt <= x <= end_zoom_dt):
        continue

    fig.add_trace(
        go.Scatter(
            x=[x, x],
            y=[ymin, ymax],
            mode="lines",
            legendgroup=SEG_GROUP,
            name="Segments" if first else None,
            showlegend=first,
            visible="legendonly",
            line=dict(color="gray", width=2, dash="dash"),  # Épaisseur déjà augmentée
            hoverinfo="skip",
        )
    )
    first = False

fig.show()

# HTML

In [41]:
import copy
import plotly.graph_objects as go

# On suppose que ta figure de base existe déjà :
# fig = ...

# 1) États (ordre : TS -> TS+SEG -> TS+SEG+FORECAST -> TS+SEG+FORECAST+CP)
states = [
    {"timeseries": True,  "forecast": False, "cp": False, "segments": False},
    {"timeseries": True,  "forecast": False, "cp": False, "segments": True},
    {"timeseries": True,  "forecast": True,  "cp": False, "segments": True},
    {"timeseries": True,  "forecast": True,  "cp": True,  "segments": True},
]

def bool_to_visible(flag: bool):
    # True  -> visible sur le graphique
    # False -> masqué mais toujours dans la légende
    return True if flag else "legendonly"

def apply_state_to_fig(f, state):
    for trace in f.data:
        name = getattr(trace, "name", "") or ""
        legendgroup = getattr(trace, "legendgroup", None)

        if "Unemployment rate" in name:
            trace.visible = bool_to_visible(state["timeseries"])
        elif "AR(1) forecast" in name:
            trace.visible = bool_to_visible(state["forecast"])
        elif "Conformal Prediction" in name:
            trace.visible = bool_to_visible(state["cp"])
        elif legendgroup == "SEGMENTS":
            trace.visible = bool_to_visible(state["segments"])
        else:
            # autres traces : ne rien toucher (comportement actuel conservé)
            pass

# 2) Frames de base (un cycle)
base_frames = []
for i, state in enumerate(states):
    fig_state = copy.deepcopy(fig)
    apply_state_to_fig(fig_state, state)
    base_frames.append(go.Frame(data=fig_state.data, name=f"state_{i}"))

# 3) Répéter le cycle
N_CYCLES = 20
frames = []
for _ in range(N_CYCLES):
    for f in base_frames:
        frames.append(go.Frame(data=f.data, name=f.name))

fig.frames = frames

# 4) S'assurer qu'il n'y a AUCUN slider résiduel
fig.layout.sliders = ()   # supprime tout slider éventuellement présent

# 5) Layout : uniquement Play / Pause
fig.update_layout(
    updatemenus=[{
        "type": "buttons",
        "showactive": True,
        "buttons": [
            {
                "label": "Play",
                "method": "animate",
                "args": [None, {
                    "frame": {"duration": 800, "redraw": True},
                    "fromcurrent": True,
                    "transition": {"duration": 300},
                    "mode": "immediate",
                }],
            },
            {
                "label": "Pause",
                "method": "animate",
                "args": [[None], {
                    "mode": "immediate",
                    "frame": {"duration": 0, "redraw": False},
                    "transition": {"duration": 0},
                }],
            },
        ],
    }],
)

fig.write_html("interactive_plot.html", include_plotlyjs="cdn")

# GIF

In [42]:
import copy
import plotly.graph_objects as go
import plotly.io as pio  # Ajouté pour exporter les images
import imageio  # Ajouté pour créer le GIF
import numpy as np  # Peut être utile pour manipuler les images si nécessaire

# On suppose que ta figure de base existe déjà :
# fig = ...

# 1) États (ordre : TS -> TS+SEG -> TS+SEG+FORECAST -> TS+SEG+FORECAST+CP)
states = [
    {"timeseries": True,  "forecast": False, "cp": False, "segments": False},
    {"timeseries": True,  "forecast": False, "cp": False, "segments": True},
    {"timeseries": True,  "forecast": True,  "cp": False, "segments": True},
    {"timeseries": True,  "forecast": True,  "cp": True,  "segments": True},
]

def bool_to_visible(flag: bool):
    # True  -> visible sur le graphique
    # False -> masqué mais toujours dans la légende
    return True if flag else "legendonly"

def apply_state_to_fig(f, state):
    for trace in f.data:
        name = getattr(trace, "name", "") or ""
        legendgroup = getattr(trace, "legendgroup", None)

        if "Unemployment rate" in name:
            trace.visible = bool_to_visible(state["timeseries"])
        elif "AR(1) forecast" in name:
            trace.visible = bool_to_visible(state["forecast"])
        elif "Conformal Prediction" in name:
            trace.visible = bool_to_visible(state["cp"])
        elif legendgroup == "SEGMENTS":
            trace.visible = bool_to_visible(state["segments"])
        else:
            # autres traces : ne rien toucher (comportement actuel conservé)
            pass

# 2) Frames de base (un cycle)
base_frames = []
for i, state in enumerate(states):
    fig_state = copy.deepcopy(fig)  # Assurez-vous que 'fig' est défini ici
    apply_state_to_fig(fig_state, state)
    base_frames.append(go.Frame(data=fig_state.data, name=f"state_{i}"))

# 3) Répéter le cycle
N_CYCLES = 20
frames = []
for _ in range(N_CYCLES):
    for f in base_frames:
        frames.append(go.Frame(data=f.data, name=f.name))

fig.frames = frames

# 4) S'assurer qu'il n'y a AUCUN slider résiduel
fig.layout.sliders = ()  # supprime tout slider éventuellement présent

# 5) Layout : uniquement Play / Pause (conservé tel quel, mais non utilisé pour le GIF)
fig.update_layout(
    updatemenus=[{
        "type": "buttons",
        "showactive": True,
        "buttons": [
            {
                "label": "Play",
                "method": "animate",
                "args": [None, {
                    "frame": {"duration": 800, "redraw": True},
                    "fromcurrent": True,
                    "transition": {"duration": 300},
                    "mode": "immediate",
                }],
            },
            {
                "label": "Pause",
                "method": "animate",
                "args": [[None], {
                    "mode": "immediate",
                    "frame": {"duration": 0, "redraw": False},
                    "transition": {"duration": 0},
                }],
            },
        ],
    }],
)

# 6) Exporter en GIF au lieu de HTML
images = []  # Liste pour stocker les bytes des images

for frame in frames:
    # Créer une figure temporaire pour ce frame
    temp_fig = go.Figure(data=frame.data, layout=fig.layout)
    
    # Exporter le frame en image PNG
    img_bytes = pio.to_image(temp_fig, format='png', width=800, height=600)  # Ajustez width et height si nécessaire
    images.append(img_bytes)  # Ajouter les bytes de l'image à la liste

# Convertir les images en GIF
imageio.mimsave('animation.gif', [imageio.imread(img) for img in images], fps=1, loop=0)  # fps=1 pour 1 image par seconde, loop=0 pour boucle infinie

print("L'animation a été exportée en GIF sous le nom 'animation.gif'.")

C:\Users\Mita\AppData\Local\Temp\ipykernel_1288\2159434845.py:100: DeprecationWarning:

Starting with ImageIO v3 the behavior of this function will switch to that of iio.v3.imread. To keep the current behavior (and make this warning disappear) use `import imageio.v2 as imageio` or call `imageio.v2.imread` directly.

d:\Portofolio Data science\Time Series\Explainable_AI_Forecast_and_explain_the_Unemployment_of_USA\venv\Lib\site-packages\imageio\plugins\pillow.py:410: DeprecationWarning:

The keyword `fps` is no longer supported. Use `duration`(in ms) instead, e.g. `fps=50` == `duration=20` (1000 * 1/50).



L'animation a été exportée en GIF sous le nom 'animation.gif'.


In [29]:
import copy
import plotly.graph_objects as go
import plotly.io as pio  # Pour exporter les images
import imageio  # Pour créer le GIF
from PIL import Image  # Pour manipuler les images
import numpy as np  # Pour les opérations sur les arrays
import io  # Pour gérer les bytes comme un flux de fichier

# On suppose que ta figure de base existe déjà :
# fig = ...

# 1) États (ordre : TS -> TS+SEG -> TS+SEG+FORECAST -> TS+SEG+FORECAST+CP)
states = [
    {"timeseries": True,  "forecast": False, "cp": False, "segments": False},
    {"timeseries": True,  "forecast": False, "cp": False, "segments": True},
    {"timeseries": True,  "forecast": True,  "cp": False, "segments": True},
    {"timeseries": True,  "forecast": True,  "cp": True,  "segments": True},
]

def bool_to_visible(flag: bool):
    return True if flag else "legendonly"

def apply_state_to_fig(f, state):
    for trace in f.data:
        name = getattr(trace, "name", "") or ""
        legendgroup = getattr(trace, "legendgroup", None)

        if "Unemployment rate" in name:
            trace.visible = bool_to_visible(state["timeseries"])
        elif "AR(1) forecast" in name:
            trace.visible = bool_to_visible(state["forecast"])
        elif "Conformal Prediction" in name:
            trace.visible = bool_to_visible(state["cp"])
        elif legendgroup == "SEGMENTS":
            trace.visible = bool_to_visible(state["segments"])
        else:
            pass

# 2) Frames de base (un cycle)
base_frames = []
for i, state in enumerate(states):
    fig_state = copy.deepcopy(fig)  # Assurez-vous que 'fig' est défini
    apply_state_to_fig(fig_state, state)
    base_frames.append(go.Frame(data=fig_state.data, name=f"state_{i}"))

# 3) Répéter le cycle
N_CYCLES = 20
frames = []
for _ in range(N_CYCLES):
    for f in base_frames:
        frames.append(go.Frame(data=f.data, name=f.name))

fig.frames = frames

# 4) Supprimer les sliders
fig.layout.sliders = ()

# 5) Exporter en GIF avec effet de fondu
original_images = []  # Liste pour stocker les images originales

for frame in frames:
    temp_fig = go.Figure(data=frame.data, layout=fig.layout)
    img_bytes = pio.to_image(temp_fig, format='png', width=800, height=600)
    original_images.append(img_bytes)  # Stocker les bytes des images

# Fonction pour appliquer l'effet de fondu entre deux images
def create_fade_transition(img1_bytes, img2_bytes, num_intermediates=5):
    # Utiliser io.BytesIO pour ouvrir les bytes comme un flux de fichier
    img1 = Image.open(io.BytesIO(img1_bytes)).convert("RGBA")
    img2 = Image.open(io.BytesIO(img2_bytes)).convert("RGBA")
    
    if img1.size != img2.size:
        img2 = img2.resize(img1.size)  # Assurer que les tailles sont identiques
    
    transition_frames = []  # Liste pour les frames intermédiaires
    
    for i in range(num_intermediates + 1):  # Inclure l'image finale
        alpha = i / num_intermediates  # Alpha de 0 à 1
        blended_img = Image.blend(img1, img2, alpha)
        transition_frames.append(blended_img)
    
    return transition_frames  # Retourne une liste d'images PIL

# Générer les frames avec transitions
final_frames = []  # Liste finale pour toutes les frames avec fondu

for i in range(len(original_images) - 1):  # Pour chaque paire d'images consécutives
    img1_bytes = original_images[i]
    img2_bytes = original_images[i + 1]
    
    transition = create_fade_transition(img1_bytes, img2_bytes, num_intermediates=5)
    for frame in transition[:-1]:  # Ajouter tous sauf le dernier (pour éviter les doublons)
        final_frames.append(frame)
    final_frames.append(transition[-1])  # Ajouter le dernier frame de la transition

# Ajouter le tout dernier frame si nécessaire
if original_images:
    final_frames.append(Image.open(io.BytesIO(original_images[-1])).convert("RGBA"))  # Utiliser io.BytesIO ici aussi

# Enregistrer le GIF
output_frames = []  # Convertir en arrays pour imageio
for pil_img in final_frames:
    output_frames.append(np.array(pil_img))  # Convertir en numpy array

imageio.mimsave('animation_with_fade.gif', output_frames, fps=2, loop=0)

print("L'animation avec effet de fondu a été exportée en GIF sous le nom 'animation_with_fade.gif'.")

L'animation avec effet de fondu a été exportée en GIF sous le nom 'animation_with_fade.gif'.


# Tracker sur MLflow

In [ ]:
# ----------------------------
# 1) Config MLflow
# ----------------------------
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Phase 1 : Before Machine Learning")

MlflowException: API request to http://127.0.0.1:5000/api/2.0/mlflow/experiments/get-by-name failed with exception HTTPConnectionPool(host='127.0.0.1', port=5000): Max retries exceeded with url: /api/2.0/mlflow/experiments/get-by-name?experiment_name=Phase+1+%3A+Before+Machine+Learning (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x000001E24ED0CB60>: Failed to establish a new connection: [WinError 10061] Aucune connexion n’a pu être établie car l’ordinateur cible l’a expressément refusée'))

In [ ]:
# =========================================================
# MLFLOW — LOGGING UNIQUEMENT (sans coverage)
# =========================================================

import os
import mlflow
import numpy as np

# ----------------------------
# Config MLflow
# ----------------------------
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Phase 1 : Before Machine Learning")

run_name = f"AR1_h{h}_bag{int(use_bagging)}_conf{int(use_conformal)}"

with mlflow.start_run(run_name=run_name):

    # ----------------------------
    # Params
    # ----------------------------
    mlflow.log_params(bundle["params"])

    # ----------------------------
    # Metrics
    # ----------------------------
    metrics = {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "n_forecasts": int(bundle["meta"]["n_forecasts"]),
        "n_intervals": int(bundle["meta"]["n_intervals_available"]),
    }

    mlflow.log_metrics(metrics)

    # ----------------------------
    # Tags (métadonnées)
    # ----------------------------
    meta_tags = {
        k: ("" if v is None else str(v))
        for k, v in bundle.get("meta", {}).items()
    }
    mlflow.set_tags(meta_tags)

    # ----------------------------
    # Artifacts
    # ----------------------------
    if os.path.exists(AR1_BUNDLE):
        mlflow.log_artifact(AR1_BUNDLE)

    if os.path.exists(AR1_LAST_META):
        mlflow.log_artifact(AR1_LAST_META)

    if os.path.exists(AR1_LAST_PKL):
        mlflow.log_artifact(AR1_LAST_PKL)

import os
import mlflow

# --- ton fig est déjà construit ici ---
# fig = ...

# 1) Sauvegarder en HTML (interactif)
FIG_HTML = "ar1_oos_plot.html"
fig.write_html(FIG_HTML, include_plotlyjs="cdn")

# 2) Logger dans MLflow (dans un run)
with mlflow.start_run(run_name=run_name):
    # ... tes params/metrics/artifacts habituels ...
    mlflow.log_artifact(FIG_HTML, artifact_path="plots")

print(f"✅ Graphique loggé sur MLflow : plots/{FIG_HTML}")

print(
    f"✅ MLflow run terminé | {run_name} | "
    f"MAE={mae:.4f} | RMSE={rmse:.4f}"
)

🏃 View run AR1_h12_bag1_conf1 at: http://127.0.0.1:5000/#/experiments/1/runs/7cc106b3e82040c2a69b96d45e2216ed
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
🏃 View run AR1_h12_bag1_conf1 at: http://127.0.0.1:5000/#/experiments/1/runs/0acab177468c4dcb96048f608c96e3a1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
✅ Graphique loggé sur MLflow : plots/ar1_oos_plot.html
✅ MLflow run terminé | AR1_h12_bag1_conf1 | MAE=0.8670 | RMSE=1.6001


In [ ]:
# =========================================================
# MLFLOW — LOGGING (sans coverage) + PLOT HTML (même run)
# =========================================================

import os
import mlflow
import numpy as np

# ----------------------------
# Config MLflow
# ----------------------------
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Phase 1 : Before Machine Learning")

run_name = f"AR1_h{h}_bag{int(use_bagging)}_conf{int(use_conformal)}"

with mlflow.start_run(run_name=run_name):

    # ----------------------------
    # Params
    # ----------------------------
    mlflow.log_params(bundle["params"])

    # ----------------------------
    # Metrics
    # ----------------------------
    metrics = {
        "MAE": float(mae),
        "RMSE": float(rmse),
        "n_forecasts": int(bundle["meta"]["n_forecasts"]),
        "n_intervals": int(bundle["meta"]["n_intervals_available"]),
    }
    mlflow.log_metrics(metrics)

    # ----------------------------
    # Tags (métadonnées)
    # ----------------------------
    meta_tags = {k: ("" if v is None else str(v)) for k, v in bundle.get("meta", {}).items()}
    mlflow.set_tags(meta_tags)

    # ----------------------------
    # Artifacts (fichiers)
    # ----------------------------
    if os.path.exists(AR1_BUNDLE):
        mlflow.log_artifact(AR1_BUNDLE)

    if os.path.exists(AR1_LAST_META):
        mlflow.log_artifact(AR1_LAST_META)

    if os.path.exists(AR1_LAST_PKL):
        mlflow.log_artifact(AR1_LAST_PKL)

    # ----------------------------
    # Artifact (graphique Plotly)
    # ----------------------------
    FIG_HTML = "ar1_oos_plot.html"
    fig.write_html(FIG_HTML, include_plotlyjs="cdn")

    # ✅ Vérification ici (juste après l'écriture)
    if not os.path.exists(FIG_HTML):
        raise FileNotFoundError(f"❌ Le fichier {FIG_HTML} n'a pas été créé. Vérifie que 'fig' existe et que l'écriture a réussi.")

    # (Optionnel) taille du fichier pour être sûr qu'il n'est pas vide
    size_bytes = os.path.getsize(FIG_HTML)
    if size_bytes < 500:  # seuil très bas, juste pour détecter un fichier quasi vide
        raise RuntimeError(f"❌ {FIG_HTML} semble vide (taille={size_bytes} bytes).")

    mlflow.log_artifact(FIG_HTML, artifact_path="plots")

print(
    f"✅ MLflow run terminé | {run_name} | "
    f"MAE={mae:.4f} | RMSE={rmse:.4f} | "
    f"Plot: plots/{FIG_HTML}"
)

🏃 View run AR1_h12_bag1_conf1 at: http://127.0.0.1:5000/#/experiments/1/runs/c9bce5ae473c47dc8bc6bc21101fb9b5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
✅ MLflow run terminé | AR1_h12_bag1_conf1 | MAE=0.8670 | RMSE=1.6001 | Plot: plots/ar1_oos_plot.html


# Graphique

Nous utilisons une méthode de Conformal Prediction séquentielle (Split CP) qui calibre l’incertitude à partir des erreurs passées observées dans des fenêtres temporelles comparables.
Les bandes sont volontairement discontinues car elles reflètent les moments où l’incertitude est statistiquement estimable.

## Est-ce que le problème vaut la peine d’être modélisé ?

### Phase 1 : 
Le taux de chômage américain suit un cycle économique d’environ 9,5 ans, correspondant au cycle long.
Une récession apparaît au début des années 1990, puis au début des années 2000 (éclatement de la bulle Internet). La crise des subprimes survient environ 8 ans plus tard, suivie de la crise du Covid-19 11 ans après. L'impact a été immense durant la crise Covid-19 mais il est vite résorbé. 

### Phase 2 : 
Le modèle **AR(1)** utilisé est volontairement très basique. Il suit essentiellement la dynamique passée du taux de chômage. Pourtant, ce **baseline simple** permet déjà de mettre en évidence des informations clés.

En période de stabilité, le modèle auto-régressif parvient globalement à **capter la direction de l’évolution du chômage**, bien qu'il reste loin des observations. Il saisit correctement la dynamique générale, même de manière approximative. Le sens de la croissance du chômage dans la crise aussi est très bien captée par ce baseline. Ces deux faits nous donnent un signal fort. La structure temporelle existe et peut être exploitée.

### Phase 3 : 
L'ajout du Conformal Prediction nous permet de nuancer ces résultats. 
Lors des **ruptures structurelles**, le comportement change nettement. Le modèle parvient à détecter les **pics de chômage**, mais la **qualité des prévisions se dégrade fortement**. Cette dégradation se reflète directement dans l’**élargissement des intervalles conformes**. Ils  traduisent une incertitude croissante du modèle.

Un point crucial apparaît alors : **la perte de la capacité d’anticipation**. Par exemple, les effets de la crise de 2008 deviennent visibles dès la seconde moitié de l’année, mais le modèle ne les capte qu’à partir d’**octobre 2009**. Le signal est détecté, mais trop tard pour une décision politique.

### En résumé : 
En résumé, le modèle capte correctement la **direction des variations du chômage**, mais sa **fiabilité diminue fortement en période de crise**. L’intérêt majeur est qu’il **ne masque pas cette incertitude**. Au contraire, il la rend visible. Cela confirme que le problème mérite d’être modélisé. 

### Direction de l'expérimentation
Si vous étiez dans le membre de l'analyste, que conseillerez-vous pour continuer la direction de l'analyse?

# Perspective 1 : Sur la justesse de l'intensité
Nous avons vu que le comportement n'est pas le même durant la stabilité et les crises. L'évaluation de la modélisation se base donc au moins sur 02 Ou 03 périodes : 
- 1990 à 2008 ; 
- Seconde moitié de 2008 à 2019 ; 
- 2019 à maintenant ; 

Pour un souci de bonne répartition du temps, nous avons donc séparer la première période en deux sous-périodes de 1990 à 2000 puis 2000 à 2008. 

# Perspective 2 : sur la capacité d'anticipation
Le modèle réagit avec retard car il n’utilise que le chômage du mois passé (Lag = 1) pour anticiper celui du mois présent. 

Faut-t-il donc augmenter le nombre de retards pris en compte par le modèle pour mieux prévoir? Ou bien certaines situations, comme les crises, relèvent-elles de chocs extérieurs, nécessitant l’intégration d’autres indicateurs que le seul historique du chômage pour être anticipées ?

Faisons alors d'une part une analyse de l'auto-corrélation du chômage pour identifier le nombre de retards optimal. 
D'autre part, analysons les crises des Etats-Unis d'amérique en profondeur. 